<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
from scipy import stats
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(df.shape)

(30000, 45)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
key_fields = ['impressions_90d','sessions_90d','word_count','content_age_days','days_since_last_update','ctr']
for col in key_fields:
    vals = df[col].dropna()
    print(col, round(vals.mean(),1), round(vals.median(),1), round(vals.skew(),2), vals.max())

impressions_90d 5200.4 731.0 11.38 517715
sessions_90d 37.1 7.0 12.13 4345
word_count 3107.8 2877.0 0.94 9546.0
content_age_days 256.2 236.0 0.49 564
days_since_last_update 46.1 20.0 1.16 373
ctr 0.5 0.1 17.44 100.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
groups = [g.dropna().values for _, g in df.groupby('trend_direction')['content_age_days']]
H, p1 = stats.kruskal(*groups)
eps1 = (H - len(groups) + 1) / (len(df) - len(groups))
print('signal 1: content_age_days vs trend_direction', round(H,2), p1, round(eps1,4), 'OPPOSITE')

pos = df.copy()
pos.loc[pos['avg_position']==0,'avg_position'] = np.nan
pos = pos.dropna(subset=['avg_position','ctr'])
pos = pos[pos['impressions_90d']>=100]
rho2, p2 = stats.spearmanr(pos['avg_position'], pos['ctr'])
print('signal 2: avg_position vs ctr', round(rho2,3), p2, 'MIXED')

sv = df.dropna(subset=['search_volume','impressions_90d'])
rho3, p3 = stats.spearmanr(sv['search_volume'], sv['impressions_90d'])
print('signal 3: search_volume vs impressions_90d', round(rho3,3), p3, 'FALSE')

signal 1: content_age_days vs trend_direction 982.03 2.7932290756170893e-211 0.0326 OPPOSITE
signal 2: avg_position vs ctr -0.362 0.0 MIXED
signal 3: search_volume vs impressions_90d -0.029 1.4003601782555241e-06 FALSE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [4]:
down = df.loc[df['is_declining_label']==1, 'days_since_last_update'].dropna()
notdown = df.loc[df['is_declining_label']==0, 'days_since_last_update'].dropna()
u, p = stats.mannwhitneyu(down, notdown, alternative='two-sided')
rank_biserial = 1 - (2*u)/(len(down)*len(notdown))
print('down mean/median', round(down.mean(),1), down.median())
print('not-down mean/median', round(notdown.mean(),1), notdown.median())
print('mann-whitney p', p, 'rank-biserial', round(rank_biserial,3))
print('flag tested: stale_visible_page assumes staleness drives decline -> MIXED, statistically detectable but practically negligible')

down mean/median 49.2 20.0
not-down mean/median 42.4 20.0
mann-whitney p 1.0655203035919358e-17 rank-biserial -0.055
flag tested: stale_visible_page assumes staleness drives decline -> MIXED, statistically detectable but practically negligible


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
print('content age has a real but small, and reversed-from-expected, relationship with decline: newer content skews down more than older content.')
print('position and ctr move together but only weakly at the individual page level; the clean group-level curve overstates how predictable one page is from its rank.')
print('staleness alone is a weak signal for decline; a content team should not rely on days_since_last_update by itself to prioritize refresh work.')

content age has a real but small, and reversed-from-expected, relationship with decline: newer content skews down more than older content.
position and ctr move together but only weakly at the individual page level; the clean group-level curve overstates how predictable one page is from its rank.
staleness alone is a weak signal for decline; a content team should not rely on days_since_last_update by itself to prioritize refresh work.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.